### write a script that reads the October export, totals spend by category, and tells us which lines it couldn't read, so we can chase those up? 

challenge -> malformed lines

1. **Field count**: exactly 4 fields after splitting on `;`. (Think about what `split(";")` returns for a line that uses `|` instead.)
2. **Date**: exactly 10 characters, a `-` at positions 4 and 7, and digits everywhere else (slicing + `isdigit`).
3. **Merchant**: not blank after cleaning.
4. **Type**: `debit` or `refund` after cleaning.
5. **Amount**: after removing `$`, `,`, spaces and a single leading `-`, what's left must be digits with at most one decimal point. `"N/A"`, `"1O.49"` (letter O) and `""` must all fail. This is the same idea as the `looks_numeric` exercise in the 2.2.2 practice notebook.

must appear in the report with 
- line number
- reason
- raw text

When you're done, deliberately break three more lines of `RAW_EXPORT` in new ways (e.g. an amount of `$12.3.4`, a date of `2026-1-05`, a lone `;;;`) and check your script reports each one correctly, then restore the original data before your final commit.

In [202]:
RAW_EXPORT = """# Corporate card export - October 2026 - generated by LegacyBank v2.3
date;merchant;amount;type
2026-10-01;  FRESHMART GROCERY ;$84.12;debit
2026-10-01;uber trip;$18.40;DEBIT
2026-10-02;Slack Technologies;$1,234.50;debit
2026-10-02;Bean There Cafe;$5.25;debit
2026-10-03;freshmart grocery;$62.07;Debit
2026-10-03;Shell Gas Station;$71.30;debit
2026-10-04;  Uber   Trip;$22.15 ;debit
2026-10-04;GitHub Inc;$21.00;debit

2026-10-05;Bean There Cafe;$4.95;debit
2026-10-05;   ;$13.00;debit
2026-10-06;Noodle House;$38.60;debit
2026-10-06;FreshMart Grocery;-$12.40;refund
2026-10-07;Adobe Systems;$54.99;debit
2026-10-07;Shell gas station;$65.80; debit
2026-10-08;Noodle House;$41.25;debit
2026-10-08;Bean There Cafe;N/A;debit
2026-10-09;Corner Market;$9.99;debit
2026/10/09;UBER TRIP;$15.75;debit
2026-10-10;Office Depot;$143.20;debit
2026-10-10;GitHub Inc;$21.00
2026-10-11;Pizza Palace;$27.50;debit
2026-10-11;Gastown Cafe;$16.80;debit
2026-10-12;Shell Gas Station;$58.10;debit
2026-10-12;Corner Market;$1O.49;debit
2026-10-13;Adobe Systems;$54.99;REFUND
2026-10-13;Transit Pass Kiosk;$156.00;debit
2026-10-14;Pizza Palace;$31.80;debit
2026-10-14;freshmart grocery;$97.45;debit

2026-10-15;Office Depot;$12.35;debit
2026-10-15|Noodle House|$22.10|debit
2026-10-16;Zoom Video;$19.99;debit
2026-10-16;Bean There Cafe;$5.25;chargeback
2026-10-17;Corner Market;$-8.00;refund
2026-10-17;Uber Trip;$19.60;debit"""

In [203]:
# cell cleaning
# skip non-data lines, split by ;, skip rows without 4 ';', skip wrong date, skip blank merchants, skip type not debit or refund, remove $, ',', 

valid = 0
lines_read = 0
skipped = 0
malformed = 0

malformed_report = []

# skip non-data lines

for line_no, line in enumerate(RAW_EXPORT.splitlines(), start=1):
    lines_read += 1

    if not line.strip() or line.strip().startswith('#') or line.lower().startswith('date;'):
        skipped += 1
        continue

    # split + field count check
    fields = line.split(';')
    if len(fields) != 4:
        malformed += 1
        malformed_report.append(f"line {line_no}: wrong field count: {line}")
        continue

    # clean each field (date, merchant, type, amount)
    date = fields[0].strip()


    merchant = fields[1].strip()
    while '  ' in merchant:
        merchant = merchant.replace('  ', ' ')
    merchant = merchant.title()


    tx_type = fields[3].lower().strip()


    amount = fields[2].replace('$','').replace(',','').replace(' ','').strip()
    is_negative = amount.startswith('-')
    if is_negative == True:
        amount = amount[1:]

    # validation -> challenge component
    # date validation
    if len(date) != 10 or date[4] != '-' or date[7] != '-' or date.replace('-','').isdigit() == False:
        malformed += 1
        malformed_report.append(f"line {line_no}: invalid date format: {line}")
        continue

    # merchant validation
    if merchant == '':
        malformed += 1
        malformed_report.append(f"line {line_no}: blank merchant: {line}")
        continue

    # type
    if tx_type != 'debit' and tx_type != 'refund':
        malformed += 1
        malformed_report.append(f"line {line_no}: invalid type format: {line}")
        continue

    # amount
    if not amount.replace('.','',1).isdigit():
        malformed += 1
        malformed_report.append(f"line {line_no}: invalid amount - non-numeric format: {line}")
        continue


    value = float(amount)
    if tx_type == 'refund':
        value = -value
    
    valid += 1


print(malformed_report)




['line 13: blank merchant: 2026-10-05;   ;$13.00;debit', 'line 19: invalid amount - non-numeric format: 2026-10-08;Bean There Cafe;N/A;debit', 'line 21: invalid date format: 2026/10/09;UBER TRIP;$15.75;debit', 'line 23: wrong field count: 2026-10-10;GitHub Inc;$21.00', 'line 27: invalid amount - non-numeric format: 2026-10-12;Corner Market;$1O.49;debit', 'line 34: wrong field count: 2026-10-15|Noodle House|$22.10|debit', 'line 36: invalid type format: 2026-10-16;Bean There Cafe;$5.25;chargeback']
